# tabular_agent — Phase 3, step 7: what a fresh agent is told

Step 6 gave the agent a way to read its own history. This step is about the fact
that it should not have to.

An **episode** is a fresh agent on the same `run_dir` — new context, same directory
— and it is the mechanism that makes a 150-experiment run survivable at all
(`0024`). The cost is that every episode wakes up knowing nothing: not the task,
not which way the metric points, not that eleven experiments have already happened
and one of them is winning.

The live runs said it *will* work that out for itself. The v5 resume prompt never
mentioned `explore_journal` and every resumed episode reached for it anyway. Then
`0025` measured what that costs:

    orientation, as a share of each episode:   26%   50%   26%   43%

Between a quarter and a half of every episode, spent re-deriving a position that
was written down on disk the whole time. And given a bigger cap the agent orients
*more*, not less — so a longer episode is not the fix.

This notebook builds the fix: a **brief**, rendered from the journal into the
episode's system prompt.

**The three properties to watch for**, because they are the whole design and two of
them are easy to lose:

1. **Rendered, never accumulated** (§4) — a pure function of the journal, so it
   cannot drift and a resumed process sees exactly what the last one would have
2. **Flat forever** (§5) — the same size at experiment 150 as at experiment 2, or
   an episode's fixed cost becomes a function of run length and the sawtooth turns
   back into a climb
3. **A starting point, not the record** (§6) — it deliberately refuses to answer
   the questions `explore_journal` answers

> Ships unexecuted. Every code cell was smoke-tested in order first; run it top to
> bottom and the numbers in the prose should be the numbers you see.

## 1. The state a fresh agent starts in

Build a run and do a little work on it — the raw table, a baseline, a real model.

In [ ]:
import os, shutil, json
import numpy as np, pandas as pd

WORK = os.path.abspath("_step7_demo")
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)

rng = np.random.default_rng(3)
n = 600
area = rng.uniform(30, 180, n)
rooms = rng.integers(1, 6, n)
district = rng.choice(["altstadt", "hafen", "vorstadt"], n)
level = {"altstadt": 2400, "hafen": 1500, "vorstadt": 900}

price = area * np.array([level[d] for d in district]) + rooms * 5000
price = price * np.exp(rng.normal(0, 0.15, n))

df = pd.DataFrame({"area": area, "rooms": rooms, "district": district, "price": price})
data_path = os.path.join(WORK, "apartments.csv")
df.to_csv(data_path, index=False)

from tabular_agent.harness.session import RunSession
from tabular_agent.harness import tools, journal
from tabular_agent.taskspec import TaskSpec

session = RunSession(os.path.join(WORK, "run"), data_path, TaskSpec(target="price"))
df.head(3)

Before anything happens, the brief is `None` — not "0 experiments so far".

That is deliberate. A run with no position has nothing to report, and a brief
announcing its own emptiness spends tokens telling the agent something the absence
of the brief already says. It also keeps a first episode byte-identical to a run
that never had episodes at all, which is what makes the two comparable when
measuring.

In [ ]:
from tabular_agent.harness.brief import render_brief

print(render_brief(session))

In [ ]:
tools.inspect_data(session)
tools.train_tabular_model(session, model_type="baseline")
tools.train_tabular_model(session, model_type="lgbm", params={"n_estimators": 80})

print(render_brief(session))

## 2. The episode boundary, in three lines

This is the whole mechanism, and it is worth doing by hand once because it looks
like nothing. A new `RunSession` on the same directory, and a new agent: no shared
object, no shared memory, no conversation carried over. `RunSession` holds only
pointers (`0003`), so the state the next episode inherits is the **directory**.

In [ ]:
# A different process would look exactly like this. Nothing is carried over.
resumed = RunSession(session.run_dir, data_path, TaskSpec(target="price"))

print("same brief from a fresh session object:", render_brief(resumed) == render_brief(session))
print("shares nothing with the original:      ", resumed is not session)

## 3. What the brief says, as the run moves under it

Every line is read off the journal at the moment of the call. Add a feature table
and the counts move; leave that table untrained and a new line appears.

In [ ]:
enc = tools.fit_encoding(session, ["district"], "target")
tools.train_tabular_model(session, table=enc["table"], model_type="lgbm",
                          params={"n_estimators": 80}, target_transform="log1p")

# An idea started and not finished — a table built with no experiment on it.
# This is exactly the trace an episode cut mid-idea leaves behind.
tools.transform_features(session, '''
def transform_features(df):
    df = df.copy()
    df["area_per_room"] = df["area"] / df["rooms"]
    return df
''', table=enc["table"])

print(render_brief(session))

Read the lines by what each is *for*:

- **task + metric direction** — the fact most likely to be assumed rather than
  checked, and assuming it inverts every comparison silently. A resumed agent has
  no other way to know: `inspect_data` reported it to somebody else.
- **best, and the table it was trained on** — `journal["best"]` is a pointer and
  does not carry the table; it is looked up from the leaderboard row, because "what
  is currently winning" is only actionable with "what it was built on".
- **budget and since-improvement** — not inferable from the leaderboard without
  also knowing the policy. An agent that thinks it has all night keeps tuning; one
  that does not know it is nearly out spends the rest of the run orienting.
- **built but never trained on** — the seam. `0024` calls episode length "a seam,
  not a count": an idea spans several calls, and a cut lands in the middle of one.
  Nothing else surfaces this, and the next agent has no memory of meaning to.

## 4. Rendered, not remembered

The first of the three properties. The brief holds no state, caches nothing, and
writes nothing — it is a pure function of `journal.load(run_dir)`.

That is `0020`'s "the state is the run directory" applied to orientation, and it is
what makes the brief impossible to *drift*. There is no second copy of the run's
position that could disagree with the journal, because there is no copy at all.

In [ ]:
before = json.dumps(journal.load(session.run_dir), sort_keys=True)

first = render_brief(session)
second = render_brief(session)

print("two calls agree:      ", first == second)
print("the run is unchanged: ", json.dumps(journal.load(session.run_dir), sort_keys=True) == before)

## 5. Flat forever — the property the design rests on

The second property, and the one that would be easiest to lose without noticing.

Fake 150 experiments straight into the journal. Synthetic on purpose: nothing in
the brief reads an artifact, so fitting 150 real models to measure a string length
would cost far more than the property is worth. The scores improve steadily so the
run never stalls — otherwise this measures the stop line flipping to its
"harness is stopping this run" branch rather than measuring growth.

In [ ]:
from tabular_agent.harness.stop import StopPolicy

session.stop_policy = StopPolicy(max_experiments=200)
short = render_brief(session)

score = 0.30
for i in range(len(journal.load(session.run_dir)["leaderboard"]) + 1, 152):
    score -= 0.001
    journal.append_experiment(session.run_dir, {
        "id": f"e{i}", "model_type": "lgbm", "params": {"n_estimators": 50},
        "target_transform": "none", "source_table": "data/work.parquet",
        "metric": "rmsle", "val_score": score, "cv_score": score,
        "fold_spread": 0.001, "features": ["area", "rooms"],
        "artifact_path": f"experiments/e{i}",
    })

long = render_brief(session)
print(long)
print()
print(f"4 experiments: {len(short):>4} chars, {len(short.splitlines())} lines")
print(f"151 experiments: {len(long):>2} chars, {len(long.splitlines())} lines")

Same lines, same order; only the counters moved. **That is the sawtooth.** If the
brief grew — even slowly — an episode's fixed cost would be a function of how long
the run had been going, and episodes would stop solving the thing they exist to
solve.

Now the counterfactual, because the temptation is real and specific. The obvious
addition to a brief is "here are the recent experiments", and it is the one thing
that must not go in:

In [ ]:
lb = journal.load(session.run_dir)["leaderboard"]

def recent(rows, k):
    return "\n".join(
        f"{r['id']}: {r['model_type']} on {r['source_table']} val {r['val_score']:.4f}"
        for r in rows[-k:]
    )

for k in (5, 20, len(lb)):
    print(f"last {k:>3} experiments as text: {len(recent(lb, k)):>6} chars"
          f"  (brief is {len(long)})")

Five is bounded and looks harmless — 214 characters, a third of the brief. It is
the thin end: five becomes ten the first time somebody wants one more fact, twenty
is already longer than the entire brief, and "all of it" is 6,388 characters of
system prompt re-sent on every turn of every episode.

The rule is easier to hold than the judgement, so it is absolute — **if a fact needs
history, it is a query, not a line in the brief.** The brief's last line says so to
the agent in as many words, and §6 is why it has to.

## 6. What it refuses to answer

The third property. The brief will not tell you which encoding paid, what a table
contains, or what was already tried on it — all of which a resumed agent genuinely
wants to know, and all of which `explore_journal` answers exactly (step 6).

`0024` tested both halves separately and the result is that they are complementary,
not alternatives:

- **`explore_journal` with no brief** — that is v5 and v6, and it is the 26–50%
  orientation tax. It works, and it is expensive.
- **A brief with no `explore_journal`** — the brief is fixed-size by construction,
  so it cannot answer anything that needs history, and the agent is back to
  re-learning by experiment what it could have looked up.

So the brief's job is to make the *first* query unnecessary, not to replace
querying. It hands over the position and points at the record.

In [ ]:
# The brief names the tool, and does not attempt its job.
print(render_brief(session).splitlines()[-1])

## 7. How it reaches the model

Wired as PydanticAI's **dynamic** system prompt — `@agent.system_prompt(dynamic=True)`,
which receives `ctx.deps` and is evaluated when the run starts rather than when the
agent object is built.

Two alternatives were rejected and both are worth knowing (`0026`). Making it a
**tool** would cost a tool call, which is exactly what is being eliminated, and a
tool can be ignored. Injecting it into the **user prompt** works, but it puts run
state in the episode driver — so every caller has to remember, and one that forgets
gets an agent that silently re-derives everything.

`TestModel` drives this without an API key: it is the wiring under test, not the
model.

In [ ]:
from pydantic_ai.models.test import TestModel
from tabular_agent.harness.agent import build_agent, Deps

def system_text(sess):
    agent = build_agent(TestModel(call_tools=[]))
    result = agent.run_sync("go", deps=Deps(session=sess))
    return "\n".join(
        part.content
        for message in result.all_messages()
        for part in message.parts
        if part.__class__.__name__ == "SystemPromptPart"
    )

fresh = RunSession(os.path.join(WORK, "run_fresh"), data_path, TaskSpec(target="price"))
print("fresh run — brief present:", "RUN STATE" in system_text(fresh))
print("this run  — brief present:", "RUN STATE" in system_text(session))

### The prompt had to change too

`SYSTEM_PROMPT` said "`inspect_data` first, **always**". That is a run-level rule,
and every episode of v5 and v6 obeyed it *per episode* — one wasted call each time,
re-deriving a task type and metric that were already in the journal. `0024` called
it the prompt's fault.

It now scopes that to a run that has not started, and points at the run state for
the rest. There is no test that can catch a regression here; it is prose, and the
only check is a live run's opening call.

In [ ]:
from tabular_agent.harness.instructions import SYSTEM_PROMPT

for line in SYSTEM_PROMPT.splitlines():
    if "inspect_data" in line or "run state" in line.lower():
        print(line)

## 8. What is not known yet

Everything above is a mechanism working as designed. Whether it *helps* is a
different claim, and it is unmeasured.

`scripts/run_house_v7_brief.py` is the experiment: a controlled repeat of v6 —
same dataset, same model, same cap, same budget, same prompts, **including the
resume prompt that still says "earlier episodes did things you have no memory of"
even though the brief now contradicts it.** Rewriting that sentence would improve
the prompt and destroy the comparison. The only difference between v6 and v7 must
be the brief.

The number to beat is v6's, counted the same way — every `explore_journal` plus
every `inspect_data` after the run's first, over the episode's tool calls:

    v6:  26%   50%   26%   43%

Three outcomes, and the middle one is the likeliest:

1. the share falls and the freed calls become experiments — the brief works
2. the share falls and the episode just ends sooner — cheaper, no more productive
3. **the share does not move** — `0026` is a well-argued no-op and should be
   marked as one in the record rather than quietly left standing

In [ ]:
# The metric, so it is legible before the run rather than after.
def orientation_share(calls, is_first_episode):
    if not calls:
        return 0.0
    inspects = calls.count("inspect_data")
    redundant = max(inspects - 1, 0) if is_first_episode else inspects
    return (calls.count("explore_journal") + redundant) / len(calls)

v6_episode2 = (["inspect_data", "explore_journal", "explore_journal"]
               + ["train_tabular_model", "explain_table", "fit_encoding"])
print(f"a v6-shaped episode: {orientation_share(v6_episode2, False):.0%} orienting")

## 9. Handing it to a model

The loop, for real. Two episodes with a small cap, so the second one is a genuine
fresh agent that has to start from the brief.

Needs `config.env` with a `REQUESTY_API_KEY`. In a notebook use `await agent.run(...)`
— `run_sync` raises inside a running event loop.

In [ ]:
from pydantic_ai.usage import UsageLimits
from pydantic_ai.exceptions import UsageLimitExceeded
from tabular_agent.harness.agent import gateway_model, DEFAULT_GATEWAY_MODEL

live = RunSession(os.path.join(WORK, "run_llm"), data_path, TaskSpec(target="price"))

PROMPTS = [
    "Build the best model you can for this dataset. Follow one idea far enough to "
    "know whether it worked, then stop and summarise. Do not call finalize_run yet.",
    "You are continuing work on this dataset that is already in progress. Earlier "
    "episodes did things you have no memory of. Carry on.",
]

for i, prompt in enumerate(PROMPTS, 1):
    print(f"\n===== EPISODE {i} " + "=" * 40)
    print("brief handed over:", (render_brief(live) or "(none — fresh run)")[:200])
    agent = build_agent(gateway_model(DEFAULT_GATEWAY_MODEL))
    try:
        result = await agent.run(prompt, deps=Deps(session=live),
                                 usage_limits=UsageLimits(tool_calls_limit=8))
        print(result.output)
    except UsageLimitExceeded:
        print("[episode cut at the cap]")

### Where this leaves the sequence

The brief is built and the episode loop is still a script. `0025` says episodes
should switch on for runs budgeted past roughly 35 experiments and stay off below
that, so moving the loop into `harness/` means deciding where that switch lives —
which is a separate decision from this one, and should not ride along with it.

After that: `val_se` (`0022`), so a run can tell *while it is running* that a gain
is inside the noise. Step 6 §7 showed the cv-vs-val gap already exceeds
`fold_spread` on every strong model in that demo, which is the same warning from
the other direction.